# AstroTalk Content Moderation — Gemini 3 Flash

Interactive runner. Each stage is its own cell:

1. **Load dataframe** — read the input CSV, build per-session message blocks
2. **Prompt caching** — create the Gemini server-side cache for the system prompt
3. **Model call** — call Gemini per session, raw output saved to JSON after every session
4. **Parsing** — parse the raw responses into structured flags
5. **Save** — write the parsed results to JSON
6. **Merge** (manual) — join flags onto the original dataframe on session_id + turn_id

Reuses the helpers in `moderate.py` / `parser.py` / `prompts.py` — no logic duplicated here.

> `MODEL_ID` and `GOOGLE_API_KEY` are hardcoded in `gemini_api.py` / `caching.py` / `moderate.py`. Requires `google-genai` and `pandas` installed.

## 0. Setup

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd

from prompts import SYSTEM_INSTRUCTION, USER_MESSAGE_TMPL
from parser import parse_llm_response
from gemini_api import call_gemini_model, MODEL_ID, GOOGLE_API_KEY  # API call only
from caching import create_gemini_cache, delete_gemini_cache       # prompt caching only
from moderate import (
    build_sessions_from_csv,
    format_session_text,
    load_results,
    save_results,
)

# Model + key are hardcoded in each module (gemini_api.py / caching.py / moderate.py)
assert GOOGLE_API_KEY and not GOOGLE_API_KEY.startswith("PASTE_"), \
    "Set GOOGLE_API_KEY in gemini_api.py / caching.py / moderate.py"
print("Model:", MODEL_ID)

# ---- Config: edit these ----
INPUT_CSV = "to_check.csv"                 # input CSV of session messages
RAW_JSON = "moderation_raw.json"           # raw model responses (incremental)
PARSED_JSON = "moderation_results.json"    # parsed structured output
SESSION_IDS = None                         # None = all sessions, or e.g. ["SESS_1001", "SESS_1003"]

## 1. Load dataframe

In [ ]:
# Raw rows as a dataframe (for browsing) ...
df = pd.read_csv(INPUT_CSV)
print(f"Rows: {len(df)}  |  Columns: {list(df.columns)}")

# ... and the grouped, cleaned per-session message blocks sent to the model
sessions = build_sessions_from_csv(Path(INPUT_CSV))
session_ids = SESSION_IDS if SESSION_IDS else list(sessions.keys())
session_ids = [sid for sid in session_ids if sid in sessions]
print(f"Sessions: {len(sessions)}  |  Selected to run: {len(session_ids)}")
df.head()

## 2. Prompt caching

Creates a server-side cache of the system prompt (1h TTL). Run once before the model-call loop; the returned `cache_name` is reused for every call.

In [ ]:
# Caching is required — the model is called via the cached system prompt only.
cache_name = create_gemini_cache(SYSTEM_INSTRUCTION)
assert cache_name, "Cache creation failed — caching is required to call the model."
print("Cache:", cache_name)

## 3. Model call — raw output updated in JSON every session

Calls Gemini per session and writes `RAW_JSON` after **every single session** (atomic). Re-running skips sessions already in the file (resume), so a failure mid-run loses nothing.

In [ ]:
raw_path = Path(RAW_JSON)
raw_results = load_results(raw_path)            # resume from whatever is already there
todo = [sid for sid in session_ids if str(sid) not in raw_results]
print(f"To call: {len(todo)}  (already done: {len(session_ids) - len(todo)})")

MAX_RETRIES = 2
for i, sid in enumerate(todo, 1):
    messages = sessions[sid]["messages"]
    user_prompt = USER_MESSAGE_TMPL.format(
        session_id=sid,
        num_messages=len(messages),
        session_text=format_session_text(messages),
    )

    raw_text, in_tok, out_tok, cache_tok, think_tok, err = "", 0, 0, 0, 0, ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_text, in_tok, out_tok, cache_tok, think_tok = call_gemini_model(
                user_prompt, cache_name
            )
            err = ""
            break
        except Exception as exc:
            err = f"{type(exc).__name__}: {exc}"
            if attempt < MAX_RETRIES:
                time.sleep(2.0)

    raw_results[str(sid)] = {
        "session_id": sid,
        "num_messages": len(messages),
        "input_tokens": in_tok,
        "output_tokens": out_tok,
        "cached_tokens": cache_tok,
        "thinking_tokens": think_tok,
        "cache_id": cache_name,
        "status": "api_error" if err else "raw",
        "error": err,
        "raw_response": raw_text,
    }
    save_results(raw_path, raw_results)          # <-- persisted every single session

    tag = "ERR " + err[:50] if err else f"{out_tok} out tok | cache={cache_tok} | think={think_tok}"
    print(f"[{i}/{len(todo)}] {sid}: {tag}")
    time.sleep(0.5)

print("Done. Raw responses in", raw_path)

### 3b. Async batch (alternative, faster)

Runs sessions **concurrently** and keeps going across prompt-cache TTL expiry (the cache is recreated automatically). Top-level `await` works directly in Jupyter. Use **either** Section 3 (sequential) **or** this cell — both write `RAW_JSON`. Tune `concurrency` down if you hit rate limits.

In [ ]:
# Alternative to Section 3: async batch — concurrent calls + self-healing cache.
# You do NOT need to run Section 2 (cell above) for this; batch.py creates and
# refreshes the cache itself, and recreates it if the TTL expires mid-run.
# Writes the same RAW_JSON, so Section 4 (parsing) works on the result either way.
from batch import run_batch

raw_results = await run_batch(
    Path(INPUT_CSV), Path(RAW_JSON),
    session_ids=session_ids, concurrency=5,
)
print("Sessions in raw file:", len(raw_results))

## 4. Parsing

Reads `RAW_JSON`, runs `parse_llm_response` on each raw response, and produces structured `session_severity` + `intents_triggered`. Parse failures are recorded, not dropped.

In [ ]:
raw_results = load_results(Path(RAW_JSON))
parsed_results = {}

for sid, entry in raw_results.items():
    out = {k: entry[k] for k in ("session_id", "num_messages", "input_tokens",
                                 "output_tokens", "cached_tokens", "thinking_tokens",
                                 "cache_id", "raw_response")
           if k in entry}
    if entry.get("status") == "api_error":
        out.update(status="api_error", error=entry.get("error", ""),
                   session_severity="", intents_triggered=[])
    else:
        try:
            p = parse_llm_response(entry.get("raw_response", ""))
            out.update(status="ok",
                       session_severity=p.get("session_severity", ""),
                       intents_triggered=p.get("intents_triggered", []))
        except Exception as exc:
            out.update(status="parse_error", error=f"{type(exc).__name__}: {exc}",
                       session_severity="", intents_triggered=[])
    parsed_results[sid] = out

ok = sum(1 for e in parsed_results.values() if e["status"] == "ok")
flagged = sum(1 for e in parsed_results.values() if e.get("intents_triggered"))
failed = sum(1 for e in parsed_results.values() if e["status"] != "ok")
print(f"Parsed {len(parsed_results)}  |  ok={ok}  flagged={flagged}  failed={failed}")

# Preview flags as a dataframe
rows = [
    {"session_id": e["session_id"], "severity": e["session_severity"],
     "turn_id": it["turn_id"], "llm_flag": it["llm_flag"], "confidence_score": it["confidence_score"]}
    for e in parsed_results.values() for it in e.get("intents_triggered", [])
]
pd.DataFrame(rows).head(20)

## 5. Save

In [ ]:
save_results(Path(PARSED_JSON), parsed_results)
print("Saved parsed results to", PARSED_JSON)

# Clean up the server-side cache when finished
if cache_name:
    delete_gemini_cache(cache_name)
    print("Deleted cache", cache_name)

## 6. Merge (manual)

Joins the parsed LLM flags back onto the original dataframe on **`session_id` + `turn_id`**, adding exactly two columns — `llm_flag` and `confidence_score`. Multiple flags on the same turn are `|`-joined. Writes `<input>_merged.csv`. Run this only after sections 3–4 have produced `parsed_results`.

In [ ]:
# Merge LLM flags onto the original dataframe, joined on (session_id, turn_id).
# Adds exactly two columns: llm_flag, confidence_score. Original rows/columns preserved.

def _norm_turn(v):
    """Normalise a turn id to a comparable string ('5', '5.0', 5 -> '5')."""
    try:
        return str(int(float(v)))
    except (ValueError, TypeError):
        return str(v).strip()

# Build (session_id, turn_id) -> [(flag, score), ...] from the parsed results
flag_map = {}
for sid, entry in parsed_results.items():
    for it in entry.get("intents_triggered", []):
        key = (str(sid).strip(), _norm_turn(it.get("turn_id")))
        flag_map.setdefault(key, []).append(
            (str(it.get("llm_flag", "")), it.get("confidence_score", ""))
        )

# Detect the turn column in the original dataframe
turn_col = "turn_id" if "turn_id" in df.columns else \
           "message_seq" if "message_seq" in df.columns else None
assert turn_col, "No turn_id / message_seq column found in df"

def _row_flags(row):
    hits = flag_map.get((str(row["session_id"]).strip(), _norm_turn(row[turn_col])), [])
    if not hits:
        return pd.Series({"llm_flag": "", "confidence_score": ""})
    return pd.Series({
        "llm_flag": "|".join(h[0] for h in hits),
        "confidence_score": "|".join(str(h[1]) for h in hits),
    })

df[["llm_flag", "confidence_score"]] = df.apply(_row_flags, axis=1)

merged_path = Path(INPUT_CSV).with_name(Path(INPUT_CSV).stem + "_merged.csv")
df.to_csv(merged_path, index=False)
print(f"Merged CSV -> {merged_path}")
print(f"Rows: {len(df)} | flagged rows: {(df['llm_flag'] != '').sum()}")
df.head(20)